<a href="https://colab.research.google.com/github/kajal-singh1/deep-learning-journey/blob/main/CatsandDogs_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())   # should print: True
print(torch.cuda.get_device_name(0))  # should print: Tesla T4

True
Tesla T4


In [ ]:
# Download the specific dataset you chose
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset -p /content/drive/MyDrive/cats_dogs_project/

# Extract it
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/cats_dogs_project/dog-and-cat-classification-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/cats_dogs_project/')

# Check what we got
!ls -la /content/drive/MyDrive/cats_dogs_project/

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 775M/775M [00:53<00:00, 15.2MB/s]

total 793704
-rw------- 1 root root 812748137 Oct  7  2023 dog-and-cat-classification-dataset.zip
drwx------ 4 root root      4096 Mar 19 01:43 PetImages


In [ ]:
# See what's inside PetImages
!ls -la /content/drive/MyDrive/cats_dogs_project/PetImages/

# Check how many images in each folder
!echo "Cat images:" && ls /content/drive/MyDrive/cats_dogs_project/PetImages/Cat/ | wc -l
!echo "Dog images:" && ls /content/drive/MyDrive/cats_dogs_project/PetImages/Dog/ | wc -l

# Look at a few file names
!ls /content/drive/MyDrive/cats_dogs_project/PetImages/Cat/ | head -5


total 8
drwx------ 2 root root 4096 Mar 19 01:43 Cat
drwx------ 2 root root 4096 Mar 19 01:45 Dog
Cat images:
12499
Dog images:
12499
0.jpg
10000.jpg
10001.jpg
10002.jpg
10003.jpg


In [ ]:
import os
from PIL import Image

def clean_dataset(folder_path):
    """Remove corrupted or non-image files"""
    cleaned = 0
    total = 0

    for filename in os.listdir(folder_path):
        total += 1
        filepath = os.path.join(folder_path, filename)

        try:
            # Try to open the image
            with Image.open(filepath) as img:
                img.verify()  # Verify it's a valid image
        except:
            # Remove corrupted files
            os.remove(filepath)
            cleaned += 1
            print(f"Removed corrupted: {filename}")

    print(f"Cleaned {cleaned} files out of {total} total")

# Clean both folders
print("Cleaning Cat images...")
clean_dataset('/content/drive/MyDrive/cats_dogs_project/PetImages/Cat/')

print("\nCleaning Dog images...")
clean_dataset('/content/drive/MyDrive/cats_dogs_project/PetImages/Dog/')

Cleaning Cat images...
Cleaned 0 files out of 12499 total

Cleaning Dog images...


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Cleaned 0 files out of 12499 total


In [ ]:
import os
import shutil
import random

def create_train_val_split(source_dir, dest_dir, val_split=0.2, max_per_class=3000):
    """Split the dataset into train/validation"""

    # Create directories
    os.makedirs(f"{dest_dir}/train/cats", exist_ok=True)
    os.makedirs(f"{dest_dir}/train/dogs", exist_ok=True)
    os.makedirs(f"{dest_dir}/val/cats", exist_ok=True)
    os.makedirs(f"{dest_dir}/val/dogs", exist_ok=True)

    # Process cats
    cat_files = [f for f in os.listdir(f"{source_dir}/Cat")
                 if f.lower().endswith(('.jpg', '.jpeg', '.png')) and f != 'Thumbs.db']
    random.shuffle(cat_files)
    cat_files = cat_files[:max_per_class]  # Limit to prevent RAM issues

    split = int(len(cat_files) * val_split)
    val_cats = cat_files[:split]
    train_cats = cat_files[split:]

    # Copy cat images
    print("Copying cat images...")
    for i, f in enumerate(train_cats):
        if i % 500 == 0:
            print(f"  Train cats: {i}/{len(train_cats)}")
        shutil.copy2(f"{source_dir}/Cat/{f}", f"{dest_dir}/train/cats/{f}")

    for i, f in enumerate(val_cats):
        if i % 100 == 0:
            print(f"  Val cats: {i}/{len(val_cats)}")
        shutil.copy2(f"{source_dir}/Cat/{f}", f"{dest_dir}/val/cats/{f}")

    # Process dogs
    dog_files = [f for f in os.listdir(f"{source_dir}/Dog")
                 if f.lower().endswith(('.jpg', '.jpeg', '.png')) and f != 'Thumbs.db']
    random.shuffle(dog_files)
    dog_files = dog_files[:max_per_class]

    split = int(len(dog_files) * val_split)
    val_dogs = dog_files[:split]
    train_dogs = dog_files[split:]

    # Copy dog images
    print("Copying dog images...")
    for i, f in enumerate(train_dogs):
        if i % 500 == 0:
            print(f"  Train dogs: {i}/{len(train_dogs)}")
        shutil.copy2(f"{source_dir}/Dog/{f}", f"{dest_dir}/train/dogs/{f}")

    for i, f in enumerate(val_dogs):
        if i % 100 == 0:
            print(f"  Val dogs: {i}/{len(val_dogs)}")
        shutil.copy2(f"{source_dir}/Dog/{f}", f"{dest_dir}/val/dogs/{f}")

    print(f"\n✓ Final counts:")
    print(f"  Train cats: {len(train_cats)}, val cats: {len(val_cats)}")
    print(f"  Train dogs: {len(train_dogs)}, val dogs: {len(val_dogs)}")

# Create the split (this will take a few minutes)
random.seed(42)
create_train_val_split(
    source_dir='/content/drive/MyDrive/cats_dogs_project/PetImages',
    dest_dir='/content/drive/MyDrive/cats_dogs_project/data',
    max_per_class=3000  # 3000 cats + 3000 dogs = 6000 total images
)

# Verify the structure
print("\n✓ Final folder structure:")
!ls -la /content/drive/MyDrive/cats_dogs_project/data/train/
!ls -la /content/drive/MyDrive/cats_dogs_project/data/val/

Copying cat images...
  Train cats: 0/2400
  Train cats: 500/2400
  Train cats: 1000/2400
  Train cats: 1500/2400
  Train cats: 2000/2400
  Val cats: 0/600
  Val cats: 100/600
  Val cats: 200/600
  Val cats: 300/600
  Val cats: 400/600
  Val cats: 500/600
Copying dog images...
  Train dogs: 0/2400
  Train dogs: 500/2400
  Train dogs: 1000/2400
  Train dogs: 1500/2400
  Train dogs: 2000/2400
  Val dogs: 0/600
  Val dogs: 100/600
  Val dogs: 200/600
  Val dogs: 300/600
  Val dogs: 400/600
  Val dogs: 500/600

✓ Final counts:
  Train cats: 2400, val cats: 600
  Train dogs: 2400, val dogs: 600

✓ Final folder structure:
total 8
drwx------ 2 root root 4096 Mar 19 02:17 cats
drwx------ 2 root root 4096 Mar 19 02:19 dogs
total 8
drwx------ 2 root root 4096 Mar 19 02:17 cats
drwx------ 2 root root 4096 Mar 19 02:19 dogs


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load datasets
trainset = ImageFolder(root='/content/drive/MyDrive/cats_dogs_project/data/train', transform=transform)
testset  = ImageFolder(root='/content/drive/MyDrive/cats_dogs_project/data/val',   transform=transform)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader  = DataLoader(testset,  batch_size=32)

print(f"✓ Train images: {len(trainset)}")      # Should be 4800
print(f"✓ Val images: {len(testset)}")         # Should be 1200
print(f"✓ Classes: {trainset.classes}")        # ['cats', 'dogs']
print(f"✓ Class mapping: {trainset.class_to_idx}")  # {'cats': 0, 'dogs': 1}

Using: cuda
✓ Train images: 4800
✓ Val images: 1200
✓ Classes: ['cats', 'dogs']
✓ Class mapping: {'cats': 0, 'dogs': 1}


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
         )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 8 *8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 1)
        )

    def forward(self, x):
      x = self.conv_layers(x)
      x = x.view(x.size(0), -1)
      x = self.fc_layers(x)
      return x

model = CNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model created and moved to GPU")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model created and moved to GPU
Model parameters: 2,190,913


In [ ]:
epochs = 15

print("Started training..")
for epoch in range(epochs):

    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch_idx, (images , labels) in enumerate(trainloader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        predicted = (torch.sigmoid(outputs) > 0.5).float()
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Batch {batch_idx}/{len(trainloader)} - Loss: {loss.item():.4f}")

    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    train_loss = running_train_loss/ len(trainloader)
    val_loss = running_val_loss / len(testloader)
    train_acc = 100 * correct_train/ total_train
    val_acc = 100 * correct_val/ total_val

    print(f"EPOCH {epoch+1}/{epochs}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f" Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print("-" * 50)

print("Training Completed")


Started training..
Epoch 1/15 - Batch 0/150 - Loss: 0.6939
Epoch 1/15 - Batch 10/150 - Loss: 0.6854
Epoch 1/15 - Batch 20/150 - Loss: 0.6647
Epoch 1/15 - Batch 30/150 - Loss: 0.6566
Epoch 1/15 - Batch 40/150 - Loss: 0.6978
Epoch 1/15 - Batch 50/150 - Loss: 0.7218
Epoch 1/15 - Batch 60/150 - Loss: 0.6854
Epoch 1/15 - Batch 70/150 - Loss: 0.6765
Epoch 1/15 - Batch 80/150 - Loss: 0.6824
Epoch 1/15 - Batch 90/150 - Loss: 0.6282
Epoch 1/15 - Batch 100/150 - Loss: 0.6846
Epoch 1/15 - Batch 110/150 - Loss: 0.6689
Epoch 1/15 - Batch 120/150 - Loss: 0.6633
Epoch 1/15 - Batch 130/150 - Loss: 0.6119
Epoch 1/15 - Batch 140/150 - Loss: 0.6386
EPOCH 1/15:
Train Loss: 0.6657 | Train Acc: 60.50%
 Val Loss: 0.5981 | Val Acc: 68.42%
--------------------------------------------------
Epoch 2/15 - Batch 0/150 - Loss: 0.5883
Epoch 2/15 - Batch 10/150 - Loss: 0.7266
Epoch 2/15 - Batch 20/150 - Loss: 0.7092
Epoch 2/15 - Batch 30/150 - Loss: 0.5554
Epoch 2/15 - Batch 40/150 - Loss: 0.6147
Epoch 2/15 - Batch 5